In [1]:
# ============================================================
# CELL 1: IMPORT THƯ VIỆN VÀ CẤU HÌNH CHUNG
# ============================================================

import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image

# Seed cố định để việc chia train/val/test có thể tái lập
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Thư mục output để lưu metadata CSV
OUTPUT_DIR = Path("/kaggle/working/metadata")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("Notebook 01 setup completed.")

OUTPUT_DIR: /kaggle/working/metadata
Notebook 01 setup completed.


In [2]:
# ============================================================
# CELL 2 - FIXED: KIỂM TRA CẤU TRÚC /kaggle/input
# ============================================================

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Danh sách cấp 1 trong /kaggle/input:")
for p in sorted(INPUT_ROOT.iterdir()):
    print("-", p.name)

print("\nDanh sách sâu hơn trong /kaggle/input:")
for p in sorted(INPUT_ROOT.rglob("*")):
    if p.is_dir():
        rel = p.relative_to(INPUT_ROOT)
        depth = len(rel.parts)
        
        # Chỉ in đến độ sâu 3 để tránh quá dài
        if depth <= 3:
            print("-", rel)

Danh sách cấp 1 trong /kaggle/input:
- competitions
- datasets

Danh sách sâu hơn trong /kaggle/input:
- competitions
- competitions/rsna-pneumonia-detection-challenge
- competitions/rsna-pneumonia-detection-challenge/stage_2_test_images
- competitions/rsna-pneumonia-detection-challenge/stage_2_train_images
- competitions/vinbigdata-chest-xray-abnormalities-detection
- competitions/vinbigdata-chest-xray-abnormalities-detection/test
- competitions/vinbigdata-chest-xray-abnormalities-detection/train
- datasets
- datasets/khanfashee
- datasets/khanfashee/nih-chest-x-ray-14-224x224-resized


In [3]:
# ============================================================
# CELL 3 - FIXED: TỰ ĐỘNG TÌM ĐƯỜNG DẪN DATASET
# Hỗ trợ cả cấu trúc:
# /kaggle/input/ten-dataset
# /kaggle/input/competitions/ten-competition
# /kaggle/input/datasets/owner/ten-dataset
# ============================================================

def find_input_dir_deep(keywords, root=INPUT_ROOT, max_depth=4):
    """
    Tìm thư mục dataset trong /kaggle/input theo keyword.
    Hàm này tìm sâu hơn để hỗ trợ Kaggle mount dạng competitions/datasets.
    
    keywords: list từ khóa, ví dụ ["rsna", "pneumonia"]
    max_depth: giới hạn độ sâu để tránh quét quá rộng
    """
    keywords = [k.lower() for k in keywords]
    candidates = []
    
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        
        rel = p.relative_to(root)
        depth = len(rel.parts)
        
        if depth > max_depth:
            continue
        
        name = str(rel).lower()
        
        if all(k in name for k in keywords):
            candidates.append(p)
    
    return candidates


# Tìm ứng viên cho từng dataset
rsna_candidates = find_input_dir_deep(["rsna", "pneumonia"])
nih_candidates = find_input_dir_deep(["nih", "chest"])
vindr_candidates = find_input_dir_deep(["vinbigdata"])

print("RSNA candidates:")
for c in rsna_candidates:
    print("-", c)

print("\nNIH candidates:")
for c in nih_candidates:
    print("-", c)

print("\nVinDr/VinBigData candidates:")
for c in vindr_candidates:
    print("-", c)


# Chọn candidate phù hợp nhất
# Với competition/dataset, thư mục cha chứa file train.csv hoặc stage_2_train_labels.csv thường là thư mục đúng.
RSNA_DIR = rsna_candidates[0] if len(rsna_candidates) > 0 else None
NIH_DIR = nih_candidates[0] if len(nih_candidates) > 0 else None
VINDR_DIR = vindr_candidates[0] if len(vindr_candidates) > 0 else None

print("\nSelected paths:")
print("RSNA_DIR :", RSNA_DIR)
print("NIH_DIR  :", NIH_DIR)
print("VINDR_DIR:", VINDR_DIR)

assert RSNA_DIR is not None, "Không tìm thấy RSNA dataset. Hãy kiểm tra Add Input."
assert NIH_DIR is not None, "Không tìm thấy NIH dataset. Hãy kiểm tra Add Input."
assert VINDR_DIR is not None, "Không tìm thấy VinBigData dataset. Hãy kiểm tra Add Input."

print("\nĐã tìm thấy đủ 3 dataset.")

RSNA candidates:
- /kaggle/input/competitions/rsna-pneumonia-detection-challenge
- /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images
- /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_test_images

NIH candidates:
- /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized
- /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224

VinDr/VinBigData candidates:
- /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection
- /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/test
- /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train

Selected paths:
RSNA_DIR : /kaggle/input/competitions/rsna-pneumonia-detection-challenge
NIH_DIR  : /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized
VINDR_DIR: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection

Đã tìm thấy đủ 3 dataset.


In [4]:
# ============================================================
# CELL 4: XEM CẤU TRÚC FILE CỦA TỪNG DATASET
# ============================================================

def show_tree(root, max_depth=2, max_items=80):
    """
    In cây thư mục rút gọn để xem cấu trúc dataset.
    max_depth: độ sâu tối đa.
    max_items: số item tối đa in ra.
    """
    root = Path(root)
    count = 0
    
    print(f"\n===== {root} =====")
    
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        
        if depth > max_depth:
            continue
        
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}- {path.name}{suffix}")
        
        count += 1
        if count >= max_items:
            print("... output truncated ...")
            break

show_tree(RSNA_DIR, max_depth=2, max_items=80)
show_tree(NIH_DIR, max_depth=2, max_items=80)
show_tree(VINDR_DIR, max_depth=2, max_items=80)


===== /kaggle/input/competitions/rsna-pneumonia-detection-challenge =====
- GCP Credits Request Link - RSNA.txt
- stage_2_detailed_class_info.csv
- stage_2_sample_submission.csv
- stage_2_test_images/
  - 0000a175-0e68-4ca4-b1af-167204a7e0bc.dcm
  - 0005d3cc-3c3f-40b9-93c3-46231c3eb813.dcm
  - 000686d7-f4fc-448d-97a0-44fa9c5d3aa6.dcm
  - 000e3a7d-c0ca-4349-bb26-5af2d8993c3d.dcm
  - 00100a24-854d-423d-a092-edcf6179e061.dcm
  - 0015597f-2d69-4bc7-b642-5b5e01534676.dcm
  - 001b0c51-c7b3-45c1-9c17-fa7594cab96e.dcm
  - 0022bb50-bf6c-4185-843e-403a9cc1ea80.dcm
  - 00271e8e-aea8-4f0a-8a34-3025831f1079.dcm
  - 0028450f-5b8e-4695-9416-8340b6f686b0.dcm
  - 002bcde0-d8da-4931-ab04-5d724e30261b.dcm
  - 002fcb77-ef76-4626-ab34-5070f15c20db.dcm
  - 003206b4-bd4a-4684-8d49-76f4cb713a30.dcm
  - 00330f7f-d114-4eb2-9c6e-558eeb3084a1.dcm
  - 00342ae8-ff81-4229-adf6-6a2ab711707b.dcm
  - 003d17f0-bd8a-485c-bc8b-daec33f53efa.dcm
  - 003dba79-1b1d-4713-add8-d72c54074f8a.dcm
  - 003ec9e3-512e-4f6e-923d-daa9f

In [5]:
# ============================================================
# CELL 5: TÌM FILE NHÃN VÀ THƯ MỤC ẢNH RSNA
# ============================================================

def find_file(root, filename):
    matches = list(Path(root).rglob(filename))
    return matches[0] if matches else None

def find_dir(root, dirname):
    matches = [p for p in Path(root).rglob(dirname) if p.is_dir()]
    return matches[0] if matches else None

RSNA_LABEL_CSV = find_file(RSNA_DIR, "stage_2_train_labels.csv")
RSNA_CLASS_INFO_CSV = find_file(RSNA_DIR, "stage_2_detailed_class_info.csv")
RSNA_TRAIN_IMG_DIR = find_dir(RSNA_DIR, "stage_2_train_images")

print("RSNA_LABEL_CSV     :", RSNA_LABEL_CSV)
print("RSNA_CLASS_INFO_CSV:", RSNA_CLASS_INFO_CSV)
print("RSNA_TRAIN_IMG_DIR :", RSNA_TRAIN_IMG_DIR)

assert RSNA_LABEL_CSV is not None, "Không tìm thấy stage_2_train_labels.csv"
assert RSNA_TRAIN_IMG_DIR is not None, "Không tìm thấy stage_2_train_images"

print("RSNA paths OK.")

RSNA_LABEL_CSV     : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
RSNA_CLASS_INFO_CSV: /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_detailed_class_info.csv
RSNA_TRAIN_IMG_DIR : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images
RSNA paths OK.


In [6]:
# ============================================================
# CELL 6: ĐỌC NHÃN RSNA
# ============================================================

rsna_labels = pd.read_csv(RSNA_LABEL_CSV)

print("Shape rsna_labels:", rsna_labels.shape)
display(rsna_labels.head())

print("\nCác cột trong rsna_labels:")
print(rsna_labels.columns.tolist())

print("\nPhân bố Target theo dòng bbox:")
print(rsna_labels["Target"].value_counts(dropna=False))

Shape rsna_labels: (30227, 6)


,patientId,x,y,width,height,Target
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0
2,00322d4d-1c29-4943-afc9-b6754be640eb,NaN,NaN,NaN,NaN,0
3,003d8fa0-6bf1-40ed-b54c-ac657f8495c5,NaN,NaN,NaN,NaN,0
4,00436515-870c-4b36-a041-de91049b9ab4,264.0,152.0,213.0,379.0,1



Các cột trong rsna_labels:
['patientId', 'x', 'y', 'width', 'height', 'Target']

Phân bố Target theo dòng bbox:
Target
0    20672
1     9555
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 7: GOM NHÃN RSNA THEO PATIENTID
# ============================================================

# Gom label: max(Target) theo patientId
rsna_grouped = (
    rsna_labels
    .groupby("patientId")
    .agg(
        label=("Target", "max"),
        num_boxes=("Target", "sum")
    )
    .reset_index()
)

# Tạo đường dẫn ảnh DICOM
rsna_grouped["image_path"] = rsna_grouped["patientId"].apply(
    lambda x: str(RSNA_TRAIN_IMG_DIR / f"{x}.dcm")
)

# Kiểm tra file có tồn tại không
rsna_grouped["exists"] = rsna_grouped["image_path"].apply(lambda x: Path(x).exists())

# Chỉ giữ ảnh tồn tại
rsna_meta = rsna_grouped[rsna_grouped["exists"]].copy()
rsna_meta = rsna_meta.drop(columns=["exists"])

print("Số ảnh RSNA sau khi gom patientId:", len(rsna_meta))
print("\nPhân bố label RSNA:")
print(rsna_meta["label"].value_counts())

display(rsna_meta.head())

Số ảnh RSNA sau khi gom patientId: 26684

Phân bố label RSNA:
label
0    20672
1     6012
Name: count, dtype: int64


,patientId,label,num_boxes,image_path
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,0,0,/kaggle/input/competitions/rsna-pneumonia-dete...
1,000924cf-0f8d-42bd-9158-1af53881a557,0,0,/kaggle/input/competitions/rsna-pneumonia-dete...
2,000db696-cf54-4385-b10b-6b16fbb3f985,1,2,/kaggle/input/competitions/rsna-pneumonia-dete...
3,000fe35a-2649-43d4-b027-e67796d412e0,1,2,/kaggle/input/competitions/rsna-pneumonia-dete...
4,001031d9-f904-4a23-b3e5-2c088acd19c6,1,2,/kaggle/input/competitions/rsna-pneumonia-dete...


In [8]:
# ============================================================
# CELL 8: LƯU BOUNDING BOX RSNA CHO PHÂN TÍCH GRAD-CAM SAU NÀY
# ============================================================

bbox_cols = ["patientId", "x", "y", "width", "height", "Target"]

rsna_bboxes = rsna_labels[bbox_cols].copy()

# Chỉ bbox thật sự có Target = 1 mới có ý nghĩa vùng viêm phổi
rsna_bboxes_pos = rsna_bboxes[rsna_bboxes["Target"] == 1].copy()

bbox_output_path = OUTPUT_DIR / "rsna_bboxes.csv"
rsna_bboxes_pos.to_csv(bbox_output_path, index=False)

print("Saved:", bbox_output_path)
print("Positive bbox rows:", len(rsna_bboxes_pos))
display(rsna_bboxes_pos.head())

Saved: /kaggle/working/metadata/rsna_bboxes.csv
Positive bbox rows: 9555


,patientId,x,y,width,height,Target
4,00436515-870c-4b36-a041-de91049b9ab4,264.0,152.0,213.0,379.0,1
5,00436515-870c-4b36-a041-de91049b9ab4,562.0,152.0,256.0,453.0,1
8,00704310-78a8-4b38-8475-49f4573b2dbb,323.0,577.0,160.0,104.0,1
9,00704310-78a8-4b38-8475-49f4573b2dbb,695.0,575.0,162.0,137.0,1
14,00aecb01-a116-45a2-956c-08d2fa55433f,288.0,322.0,94.0,135.0,1


In [9]:
# ============================================================
# CELL 9: CHIA RSNA TRAIN / VAL / TEST
# ============================================================

# Chia train_temp và test trước: test = 15%
rsna_train_val, rsna_test = train_test_split(
    rsna_meta,
    test_size=0.15,
    random_state=SEED,
    stratify=rsna_meta["label"]
)

# Từ phần còn lại, chia val = 15% tổng.
# Vì train_val đang là 85%, val cần chiếm 15/85 = 0.1765 của train_val.
val_ratio_from_train_val = 0.15 / 0.85

rsna_train, rsna_val = train_test_split(
    rsna_train_val,
    test_size=val_ratio_from_train_val,
    random_state=SEED,
    stratify=rsna_train_val["label"]
)

print("RSNA split sizes:")
print("Train:", len(rsna_train))
print("Val  :", len(rsna_val))
print("Test :", len(rsna_test))

print("\nLabel distribution:")
print("Train:\n", rsna_train["label"].value_counts(normalize=True))
print("Val:\n", rsna_val["label"].value_counts(normalize=True))
print("Test:\n", rsna_test["label"].value_counts(normalize=True))

RSNA split sizes:
Train: 18678
Val  : 4003
Test : 4003

Label distribution:
Train:
 label
0    0.774708
1    0.225292
Name: proportion, dtype: float64
Val:
 label
0    0.774669
1    0.225331
Name: proportion, dtype: float64
Test:
 label
0    0.774669
1    0.225331
Name: proportion, dtype: float64


In [10]:
# ============================================================
# CELL 10: LƯU METADATA RSNA
# ============================================================

rsna_train_path = OUTPUT_DIR / "rsna_train.csv"
rsna_val_path = OUTPUT_DIR / "rsna_val.csv"
rsna_test_path = OUTPUT_DIR / "rsna_test.csv"

rsna_train.to_csv(rsna_train_path, index=False)
rsna_val.to_csv(rsna_val_path, index=False)
rsna_test.to_csv(rsna_test_path, index=False)

print("Saved:")
print(rsna_train_path)
print(rsna_val_path)
print(rsna_test_path)

Saved:
/kaggle/working/metadata/rsna_train.csv
/kaggle/working/metadata/rsna_val.csv
/kaggle/working/metadata/rsna_test.csv


In [11]:
# ============================================================
# CELL 11: TÌM TOÀN BỘ ẢNH NIH CHO PRETRAINING
# ============================================================

image_exts = [".png", ".jpg", ".jpeg"]

nih_image_paths = []
for ext in image_exts:
    nih_image_paths.extend(list(Path(NIH_DIR).rglob(f"*{ext}")))

nih_image_paths = sorted([str(p) for p in nih_image_paths])

print("Số ảnh NIH tìm được:", len(nih_image_paths))
print("Ví dụ 5 ảnh đầu:")
for p in nih_image_paths[:5]:
    print(p)

assert len(nih_image_paths) > 0, "Không tìm thấy ảnh NIH. Hãy kiểm tra cấu trúc dataset."

Số ảnh NIH tìm được: 112120
Ví dụ 5 ảnh đầu:
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224/00000001_000.png
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224/00000001_001.png
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224/00000001_002.png
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224/00000002_000.png
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224/00000003_000.png


In [12]:
# ============================================================
# CELL 12: TẠO NIH METADATA 30K / 50K
# ============================================================

nih_meta = pd.DataFrame({
    "image_path": nih_image_paths
})

# Shuffle cố định
nih_meta = nih_meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_30k = min(30000, len(nih_meta))
n_50k = min(50000, len(nih_meta))

nih_30k = nih_meta.iloc[:n_30k].copy()
nih_50k = nih_meta.iloc[:n_50k].copy()

nih_30k_path = OUTPUT_DIR / "nih_pretrain_30k.csv"
nih_50k_path = OUTPUT_DIR / "nih_pretrain_50k.csv"

nih_30k.to_csv(nih_30k_path, index=False)
nih_50k.to_csv(nih_50k_path, index=False)

print("NIH total images:", len(nih_meta))
print("Saved 30k subset:", nih_30k_path, "Rows:", len(nih_30k))
print("Saved 50k subset:", nih_50k_path, "Rows:", len(nih_50k))

display(nih_50k.head())

NIH total images: 112120
Saved 30k subset: /kaggle/working/metadata/nih_pretrain_30k.csv Rows: 30000
Saved 50k subset: /kaggle/working/metadata/nih_pretrain_50k.csv Rows: 50000


,image_path
0,/kaggle/input/datasets/khanfashee/nih-chest-x-...
1,/kaggle/input/datasets/khanfashee/nih-chest-x-...
2,/kaggle/input/datasets/khanfashee/nih-chest-x-...
3,/kaggle/input/datasets/khanfashee/nih-chest-x-...
4,/kaggle/input/datasets/khanfashee/nih-chest-x-...


In [13]:
# ============================================================
# CELL 13: KIỂM TRA ẢNH NIH CÓ ĐỌC ĐƯỢC KHÔNG
# ============================================================

sample_nih_paths = nih_50k["image_path"].sample(
    n=min(5, len(nih_50k)), 
    random_state=SEED
).tolist()

for p in sample_nih_paths:
    img = Image.open(p)
    print(Path(p).name, "mode:", img.mode, "size:", img.size)

00019369_007.png mode: L size: (224, 224)
00011956_006.png mode: L size: (224, 224)
00009604_005.png mode: L size: (224, 224)
00001200_001.png mode: L size: (224, 224)
00003973_009.png mode: L size: (224, 224)


In [14]:
# ============================================================
# CELL 14: TÌM FILE NHÃN VÀ THƯ MỤC ẢNH VINDR/VINBIGDATA
# ============================================================

VINDR_TRAIN_CSV = find_file(VINDR_DIR, "train.csv")
VINDR_TRAIN_IMG_DIR = find_dir(VINDR_DIR, "train")

print("VINDR_TRAIN_CSV    :", VINDR_TRAIN_CSV)
print("VINDR_TRAIN_IMG_DIR:", VINDR_TRAIN_IMG_DIR)

assert VINDR_TRAIN_CSV is not None, "Không tìm thấy train.csv của VinDr/VinBigData"
assert VINDR_TRAIN_IMG_DIR is not None, "Không tìm thấy thư mục train của VinDr/VinBigData"

print("VinDr paths OK.")

VINDR_TRAIN_CSV    : /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv
VINDR_TRAIN_IMG_DIR: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train
VinDr paths OK.


In [15]:
# ============================================================
# CELL 15: ĐỌC NHÃN VINDR/VINBIGDATA
# ============================================================

vindr_df = pd.read_csv(VINDR_TRAIN_CSV)

print("Shape vindr_df:", vindr_df.shape)
display(vindr_df.head())

print("\nCác cột trong VinDr:")
print(vindr_df.columns.tolist())

print("\nPhân bố class_name:")
print(vindr_df["class_name"].value_counts())

Shape vindr_df: (67914, 8)


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN



Các cột trong VinDr:
['image_id', 'class_name', 'class_id', 'rad_id', 'x_min', 'y_min', 'x_max', 'y_max']

Phân bố class_name:
class_name
No finding            31818
Aortic enlargement     7162
Cardiomegaly           5427
Pleural thickening     4842
Pulmonary fibrosis     4655
Nodule/Mass            2580
Lung Opacity           2483
Pleural effusion       2476
Other lesion           2203
Infiltration           1247
ILD                    1000
Calcification           960
Consolidation           556
Atelectasis             279
Pneumothorax            226
Name: count, dtype: int64


In [16]:
# ============================================================
# CELL 16: MAPPING VINDR SANG BINARY PNEUMONIA / NO FINDING
# FIX: VinDr không có class "Pneumonia" — dùng findings tương đương
# ============================================================

vindr_df["class_name_clean"] = vindr_df["class_name"].astype(str).str.strip()

# FIX: Pneumonia findings trong VinDr
# "Lung Opacity" + "Consolidation" + "Infiltration" là các biểu hiện
# X-quang của viêm phổi được radiologist VinDr dùng
PNEUMONIA_EQUIV = {"Lung Opacity", "Consolidation", "Infiltration"}

pneumonia_ids = set(
    vindr_df.loc[
        vindr_df["class_name_clean"].isin(PNEUMONIA_EQUIV), "image_id"
    ].unique()
)

# Chỉ lấy "No finding" thuần túy — không overlap với pneumonia
no_finding_ids = set(
    vindr_df.loc[
        vindr_df["class_name_clean"] == "No finding", "image_id"
    ].unique()
) - pneumonia_ids

print(f"Pneumonia IDs : {len(pneumonia_ids):,}")
print(f"No finding IDs: {len(no_finding_ids):,}")

# Phần còn lại giữ nguyên
vindr_pos = pd.DataFrame({"image_id": list(pneumonia_ids), "label": 1})
vindr_neg = pd.DataFrame({"image_id": list(no_finding_ids), "label": 0})

vindr_external_meta = pd.concat([vindr_pos, vindr_neg], ignore_index=True)

def find_vindr_image_path(image_id):
    possible_exts = [".dicom", ".dcm", ".png", ".jpg", ".jpeg"]
    for ext in possible_exts:
        p = VINDR_TRAIN_IMG_DIR / f"{image_id}{ext}"
        if p.exists():
            return str(p)
    return None

vindr_external_meta["image_path"] = vindr_external_meta["image_id"].apply(find_vindr_image_path)
vindr_external_meta = vindr_external_meta[vindr_external_meta["image_path"].notna()].copy()
vindr_external_meta = vindr_external_meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"\nVinDr external binary size: {len(vindr_external_meta):,}")
print(vindr_external_meta["label"].value_counts())
display(vindr_external_meta.head())

Pneumonia IDs : 1,588
No finding IDs: 10,606

VinDr external binary size: 12,194
label
0    10606
1     1588
Name: count, dtype: int64


,image_id,label,image_path
0,8ceb7a01369d1f273deb2879db8ad1d8,0,/kaggle/input/competitions/vinbigdata-chest-xr...
1,209ee16958c3644d75cc54865d5b7067,0,/kaggle/input/competitions/vinbigdata-chest-xr...
2,95d225169091ca8b9806f685af6c8071,0,/kaggle/input/competitions/vinbigdata-chest-xr...
3,0370b082971fe47663fedaf76fa5e492,0,/kaggle/input/competitions/vinbigdata-chest-xr...
4,ae5cec1517ab3e82c5374e4c6219a17d,1,/kaggle/input/competitions/vinbigdata-chest-xr...


In [17]:
# ============================================================
# CELL 17: LƯU METADATA VINDR EXTERNAL VALIDATION
# ============================================================

vindr_external_path = OUTPUT_DIR / "vindr_external_test.csv"
vindr_external_meta.to_csv(vindr_external_path, index=False)

print("Saved:", vindr_external_path)

Saved: /kaggle/working/metadata/vindr_external_test.csv


In [18]:
# ============================================================
# CELL 18: TẠO BẢNG THỐNG KÊ DATASET
# ============================================================

stats = []

# NIH
stats.append({
    "dataset": "NIH ChestX-ray14 224x224",
    "role": "Self-supervised pretraining",
    "split": "pretrain_30k",
    "num_images": len(nih_30k),
    "num_positive": None,
    "num_negative": None,
    "note": "Unlabeled images for I-JEPA pretraining"
})

stats.append({
    "dataset": "NIH ChestX-ray14 224x224",
    "role": "Self-supervised pretraining",
    "split": "pretrain_50k",
    "num_images": len(nih_50k),
    "num_positive": None,
    "num_negative": None,
    "note": "Main pretraining subset"
})

# RSNA
for split_name, df in [
    ("train", rsna_train),
    ("val", rsna_val),
    ("test", rsna_test)
]:
    stats.append({
        "dataset": "RSNA Pneumonia Detection",
        "role": "Fine-tuning / internal evaluation",
        "split": split_name,
        "num_images": len(df),
        "num_positive": int((df["label"] == 1).sum()),
        "num_negative": int((df["label"] == 0).sum()),
        "note": "Binary pneumonia classification"
    })

# VinDr
stats.append({
    "dataset": "VinBigData/VinDr-CXR",
    "role": "External validation",
    "split": "external_test",
    "num_images": len(vindr_external_meta),
    "num_positive": int((vindr_external_meta["label"] == 1).sum()),
    "num_negative": int((vindr_external_meta["label"] == 0).sum()),
    "note": "Only Pneumonia and No finding images are used"
})

dataset_statistics = pd.DataFrame(stats)
stats_path = OUTPUT_DIR / "dataset_statistics.csv"
dataset_statistics.to_csv(stats_path, index=False)

display(dataset_statistics)

print("Saved:", stats_path)

,dataset,role,split,num_images,num_positive,num_negative,note
0,NIH ChestX-ray14 224x224,Self-supervised pretraining,pretrain_30k,30000,NaN,NaN,Unlabeled images for I-JEPA pretraining
1,NIH ChestX-ray14 224x224,Self-supervised pretraining,pretrain_50k,50000,NaN,NaN,Main pretraining subset
2,RSNA Pneumonia Detection,Fine-tuning / internal evaluation,train,18678,4208.0,14470.0,Binary pneumonia classification
3,RSNA Pneumonia Detection,Fine-tuning / internal evaluation,val,4003,902.0,3101.0,Binary pneumonia classification
4,RSNA Pneumonia Detection,Fine-tuning / internal evaluation,test,4003,902.0,3101.0,Binary pneumonia classification
5,VinBigData/VinDr-CXR,External validation,external_test,12194,1588.0,10606.0,Only Pneumonia and No finding images are used


Saved: /kaggle/working/metadata/dataset_statistics.csv


In [19]:
# ============================================================
# CELL 19: KIỂM TRA FILE OUTPUT
# ============================================================

print("Các file đã tạo trong OUTPUT_DIR:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

Các file đã tạo trong OUTPUT_DIR:
- dataset_statistics.csv
- nih_pretrain_30k.csv
- nih_pretrain_50k.csv
- rsna_bboxes.csv
- rsna_test.csv
- rsna_train.csv
- rsna_val.csv
- vindr_external_test.csv


In [20]:
# ============================================================
# CELL 20: NÉN METADATA THÀNH ZIP
# ============================================================

zip_path = "/kaggle/working/metadata_notebook01"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

print("Created zip:", zip_path + ".zip")

Created zip: /kaggle/working/metadata_notebook01.zip
